In [53]:
from dataclasses import dataclass
from typing import List, Dict, Tuple
import numpy as np
import scipy.linalg

from qiskit.circuit import QuantumCircuit, QuantumRegister, AncillaRegister
from qiskit.quantum_info import Operator
from qiskit.synthesis import OneQubitEulerDecomposer
from qiskit.circuit.library import UnitaryGate, GlobalPhaseGate  

from decomposer import GateSynthesizer, TwoQubitDecomposer, ThreeQubitDecomposer, PhaseCorrector

from utils import is_energy_conserving, get_hamming_weight_blocks, extract_two_level_blocks, get_nontrivial_rows_cols, print_nontrivial_unitary_basis_action, get_effective_unitary, _unitaries_close_up_to_phase
from utils import ControlledTwoLevel, TwoLevelUnitary

def V_su2(a: complex, b: complex) -> np.ndarray:
    """
    Construct V(a,b) ∈ SU(2) such that V(a,b) @ [a, b]^T = [sqrt(|a|²+|b|²), 0]^T.
    
    From Eq. (D1):
        V(a,b) = (|a|² + |b|²)^{-1/2} * [[a*, b*], [-b, a]]
    """
    norm_sq = np.abs(a)**2 + np.abs(b)**2
    
    if norm_sq < 1e-14:
        return np.eye(2, dtype=complex)
    
    norm = np.sqrt(norm_sq)
    V = np.array([
        [np.conj(a), np.conj(b)],
        [-b, a]
    ], dtype=complex) / norm
    
    return V

def verify_sud_decomposition(U: np.ndarray, gates: List[TwoLevelUnitary], tol: float = 1e-10) -> bool:
    """Verify that the product of gates equals U."""
    d = U.shape[0]
    product = np.eye(d, dtype=complex)
    
    for gate in gates:
        product = product @ gate.to_full_matrix()
    
    if np.allclose(product, U, atol=tol):
        return True
    else:
        print(f"Max error: {np.max(np.abs(product - U))}")
        with np.printoptions(precision=3, suppress=True):
            print("Expected U:")
            print(U)
            print("Got product:")
            print(product)
        return False

def sud_decompose(U: np.ndarray, tol: float = 1e-12) -> List[TwoLevelUnitary]:
    d = U.shape[0]
    
    # Validate input
    if U.shape != (d, d):
        raise ValueError("Input must be a square matrix")
    if not np.allclose(U @ U.conj().T, np.eye(d), atol=tol):
        raise ValueError("Input matrix is not unitary")
    if not np.isclose(np.linalg.det(U), 1.0, atol=tol):
        raise ValueError(f"Input matrix is not special unitary (det = {np.linalg.det(U)})")
    
    # return empty list for identity
    if np.allclose(U, np.eye(d), atol=tol):
        return []
    # Base case: d = 2
    if d == 2:
        return [TwoLevelUnitary(dim=2, i=0, j=1, submatrix=U.copy())]
    
    # Recursive case: d > 2
    W = U.copy().astype(complex)
    v_gates: List[TwoLevelUnitary] = []
    
    # We want to transform first column [a_1, a_2, ..., a_d]^T to [1, 0, ..., 0]^T
    # 
    # Step 1: Apply V(a_1, a_2)_{0,1} to get [sqrt(|a_1|^2+|a_2|^2), 0, a_3, ..., a_d]^T
    # Step 2: Apply V(W[0,0], a_3)_{0,2} to zero out position 2
    # ... and so on
    #
    # After each V application, W[0,0] contains the accumulated norm
    
    for k in range(1, d):
        # Current state: W[0,0] has accumulated value, W[k,0] is to be zeroed
        a = W[0, 0]  # The accumulated value at position 0
        b = W[k, 0]  # The value to zero out
        
        # Build V(a, b) acting on subspace {0, k}
        V_2x2 = V_su2(a, b)
        
        # Create 2-level unitary
        V_gate = TwoLevelUnitary(dim=d, i=0, j=k, submatrix=V_2x2)
        v_gates.append(V_gate)
        
        # Apply V to W (left multiply): affects rows 0 and k
        # More efficient than full matrix multiplication
        row_0 = W[0, :].copy()
        row_k = W[k, :].copy()
        
        W[0, :] = V_2x2[0, 0] * row_0 + V_2x2[0, 1] * row_k
        W[k, :] = V_2x2[1, 0] * row_0 + V_2x2[1, 1] * row_k
    
    # Now W should have the form [[1, 0...], [0, W']] where W' ∈ SU(d-1)
    # Verify
    if not np.isclose(np.abs(W[0, 0]), 1.0, atol=tol):
        raise RuntimeError(f"Failed to reduce: |W[0,0]| = {np.abs(W[0,0])}")
    if not np.allclose(W[1:, 0], 0, atol=tol):
        raise RuntimeError(f"Failed to zero first column")
    if not np.allclose(W[0, 1:], 0, atol=tol):
        raise RuntimeError(f"First row not zeroed")
    
    # Handle potential phase on W[0,0] - it should be 1, but might be e^{iφ}
    # Since W ∈ SU(d), if first row and column are [1,0,...] and [1,0,...]^T,
    # then W[0,0] must be 1 (not just |W[0,0]|=1)
    phase = W[0, 0]
    if not np.isclose(phase, 1.0, atol=tol):
        # This shouldn't happen for SU(d), but handle it
        print(f"Warning: W[0,0] = {phase}, expected 1.0")
    
    # Extract the (d-1)×(d-1) block
    W_sub = W[1:, 1:].copy()
    
    # Verify W_sub is in SU(d-1)
    det_W_sub = np.linalg.det(W_sub)
    if not np.isclose(det_W_sub, 1.0, atol=tol):
        # The determinant might be off by the phase we extracted
        # For proper SU(d), this should be 1
        print(f"Warning: det(W_sub) = {det_W_sub}")
    
    # Recursively decompose W_sub
    sub_gates = sud_decompose(W_sub, tol=tol)
    
    # Embed sub_gates into d-dimensional space (shift indices by 1)
    embedded_sub_gates: List[TwoLevelUnitary] = []
    for gate in sub_gates:
        # skip identity gates
        if np.allclose(gate.submatrix, np.eye(2), atol=tol):
            continue
        
        embedded = TwoLevelUnitary(
            dim=d,
            i=gate.i + 1,
            j=gate.j + 1,
            submatrix=gate.submatrix.copy()
        )
        embedded_sub_gates.append(embedded)
    
    if not embedded_sub_gates:
        print("Warning: No non-trivial sub-gates extracted from W_sub")
    # Reconstruct U:
    # We applied: V_{d-1} @ ... @ V_1 @ U = 1 ⊕ W_sub
    # where V_k = V_{0,k+1} in 0-indexed notation
    #
    # So: U = V_1† @ V_2† @ ... @ V_{d-1}† @ (1 ⊕ W_sub)
    #
    # v_gates = [V_1, V_2, ..., V_{d-1}] in order of application
    # We need [V_1†, V_2†, ..., V_{d-1}†, sub_gates...]
    
    result_gates: List[TwoLevelUnitary] = []
    
    # V gates need to be inverted (daggered)
    # The order: we applied V_{d-1} @ ... @ V_1 @ U
    # So U = V_1† @ V_2† @ ... @ V_{d-1}† @ (1 ⊕ W)
    for V_gate in v_gates:
        result_gates.append(V_gate.dagger())
    
    # Add the embedded sub-gates
    result_gates.extend(embedded_sub_gates)
    
    # assert that the prodcut of the result_gates equals U
    assert verify_sud_decomposition(U, result_gates), "Decomposition verification failed"
    
    return result_gates

def apply_controlled_iswap(qc: QuantumCircuit, control: int, target1: int, target2: int, 
                        control_val: int = 1, inverse: bool = False):
    """Apply a controlled-iSWAP or controlled-iSWAP† gate.
    
    Args:
        qc: QuantumCircuit
        control: Control qubit index
        target1: First target qubit
        target2: Second target qubit
        control_val: Control value (0 or 1)
        inverse: If True, apply iSWAP† (controlled-iSWAP†)
    """
    from qiskit.circuit.library import iSwapGate
    
    # If control_val is 0, flip the control qubit
    if control_val == 0:
        qc.x(control)
    
    # Create controlled-iSWAP
    if inverse:
        gate = iSwapGate().inverse().control(1)
    else:
        gate = iSwapGate().control(1)
    
    qc.append(gate, [control, target1, target2])
    
    if control_val == 0:
        qc.x(control)

def construct_conjugation_gates(b: int, b_prime: int, n_qubits: int) -> Tuple[List[Tuple[int, int, int]], int]:
    """Construct the sequence of controlled-iSWAP gates K that conjugates
    a 2-level unitary from (b, b') to (b'', b') where d(b'', b') = 2.
    
    Following the construction in Lemma 10 of arXiv:2309.11051.
    
    Args:
        b: First basis state index
        b_prime: Second basis state index  
        n_qubits: Number of qubits
        
    Returns:
        Tuple of (gates_list, b_double_prime) where:
        - gates_list: List of (control_bit, target_bit1, target_bit2) for controlled-iSWAPs
        - b_double_prime: The intermediate state index with d(b_double_prime, b_prime) = 2
    """
    # Work with bit indices consistently using bit-indexing (LSB = bit 0)
    # Extract bits using bit operations, not string operations
    
    # Find positions where b and b' differ (using bit indexing)
    diff_positions = []
    for bit_idx in range(n_qubits):
        bit_b = (b >> bit_idx) & 1
        bit_b_prime = (b_prime >> bit_idx) & 1
        if bit_b != bit_b_prime:
            diff_positions.append(bit_idx)
    
    hamming_dist = len(diff_positions)
    
    if hamming_dist % 2 != 0:
        raise ValueError("Hamming distance must be even for equal Hamming weight states")
    
    t = hamming_dist // 2  # Number of swaps needed
    
    if t <= 1:
        # Already at distance 2 or 0
        return [], b
    
    # Partition differing positions into l_j (1 in b, 0 in b') and r_j (0 in b, 1 in b')
    l_positions = []  # positions where b has 1 and b' has 0
    r_positions = []  # positions where b has 0 and b' has 1
    
    for bit_idx in diff_positions:
        bit_b = (b >> bit_idx) & 1
        bit_b_prime = (b_prime >> bit_idx) & 1
        if bit_b == 1 and bit_b_prime == 0:
            l_positions.append(bit_idx)
        elif bit_b == 0 and bit_b_prime == 1:
            r_positions.append(bit_idx)
    
    if len(l_positions) != t or len(r_positions) != t:
        raise ValueError(f"Partition error: found {len(l_positions)} l's and {len(r_positions)} r's, expected {t} each")
    
    # Helper function to format state as binary string for display
    def format_state(state_int):
        return bin(state_int)[2:].zfill(n_qubits)
    
    print(f"    Constructing conjugation sequence:")
    print(f"      b  = |{format_state(b)}⟩ = {b}")
    print(f"      b' = |{format_state(b_prime)}⟩ = {b_prime}")
    print(f"      t = {t} (need {t-1} controlled-iSWAPs)")
    print(f"      l_positions (1→0): {l_positions}")
    print(f"      r_positions (0→1): {r_positions}")
    
    # Build sequence of controlled-iSWAP gates
    # Following Eq. (69)-(72) from the paper
    gates = []
    current_state = b  # Start from b
    
    for j in range(t - 1):  # j = 0, 1, ..., t-2 (we need t-1 gates)
        # print(f"      Step {j+1}/{t-1}: Current state: |{format_state(current_state)}⟩")
        lj = l_positions[j]
        rj = r_positions[j]
        
        # Control bit: l_{j+1}
        # This must be a bit where:
        # - current state has 1 (so gate acts)
        # - b' has 0 (so gate doesn't act on b')
        control_bit = l_positions[j + 1]
        
        print(f"      Gate {j}: Controlled-iSWAP with control={control_bit}, targets=({lj},{rj})")
        print(f"        Current state: |{format_state(current_state)}⟩")
        
        # Verify control bit is 1 in current state and 0 in b'
        current_control_bit = (current_state >> control_bit) & 1
        bprime_control_bit = (b_prime >> control_bit) & 1
        
        print(f"        Control bit {control_bit}: current={current_control_bit}, b'={bprime_control_bit}")
        
        if current_control_bit != 1:
            raise ValueError(f"Control bit {control_bit} should be 1 in current state")
        if bprime_control_bit != 0:
            raise ValueError(f"Control bit {control_bit} should be 0 in b' for gate to not act on b'")
        
        gates.append((control_bit, lj, rj))
        
        # Update current state by swapping bits at lj and rj
        # Extract the two bits
        bit_lj = (current_state >> lj) & 1
        bit_rj = (current_state >> rj) & 1
        
        # Swap them
        if bit_lj != bit_rj:  # Only need to swap if they're different
            # Clear both bits
            current_state &= ~(1 << lj)
            current_state &= ~(1 << rj)
            # Set them to swapped values
            current_state |= (bit_rj << lj)
            current_state |= (bit_lj << rj)
    
    b_double_prime = current_state
    
    print(f"      Final state b'': |{format_state(b_double_prime)}⟩ = {b_double_prime}")
    
    # Verify the Hamming distance
    hd_final = bin(b_double_prime ^ b_prime).count('1')
    print(f"      Hamming distance d(b'', b'): {hd_final}")
    
    if hd_final != 2:
        raise ValueError(f"Final Hamming distance should be 2, got {hd_final}")
    
    return gates, b_double_prime

def fix_relative_phases(qc: QuantumCircuit, qargs: List[int], phases: List[float]):
    """Fix relative phases between different Hamming weight sectors.
    
    Args:
        qc: Quantum circuit
        qargs: List of logical qubit indices
        phases: List of phases [theta_0, theta_1, ...] for each HW sector
    """
    theta_0 = phases[0]
    num_qubits = len(qargs)
    print(f"\nFixing relative phases: {phases}")
    
    # Track if we've added an ancilla (do this OUTSIDE the loop)
    anc = None
    
    for m, theta in enumerate(phases[1:], start=1):  # start=1 to get correct HW
        delta = theta - theta_0
        
        if np.abs(delta) < 1e-10:
            continue  # Skip if no phase correction needed
        print(f"\nFixing relative phase for Hamming weight {m}: delta = {delta:.4f} rad")
        # Add ancilla only once, when first needed
        if anc is None:
            if qc.num_ancillas > 0:
                # Ancilla already exists - get its index
                # The ancilla qubit index in the circuit
                anc = qc.num_qubits - qc.num_ancillas  # First ancilla index
            else:
                # Add new ancilla register
                qc.add_register(AncillaRegister(1, 'ancilla'))
                anc = qc.num_qubits - 1  # Last qubit is the ancilla
            
            # Add ancilla to qargs for the decomposition
            qargs_with_anc = qargs + [anc]
        else:
            qargs_with_anc = qargs + [anc]
        
        # Step 1: Pick |b>: any bitstring of weight m (not m+1, since enumerate starts at 1)
        hw_basis_list = []
        for i in range(2**num_qubits):
            bits = bin(i)[2:].zfill(num_qubits)
            if bits.count('1') == m:
                hw_basis_list.append(bits)
                

        if not hw_basis_list:
            raise ValueError(f"No basis state with Hamming weight {m}")
        
        b = hw_basis_list[0]  # Pick the first one for simplicity
        # Step 2: Construct |b'> by flipping one '1' → '0'
        b_list = list(b)
        flip_index = None
        for idx in range(num_qubits):
            if b_list[idx] == '1':
                b_list[idx] = '0'
                flip_index = idx
                break

        b_prime = ''.join(b_list)
        
        print(f" |b> = |{b}>, |b'> = |{b_prime}>")

        # Step 3: Embed into (n+1)-qubit Hilbert space
        # Convention: ancilla is the MOST significant bit (leftmost)
        dim = 2**(num_qubits + 1)
        H = np.zeros((dim, dim), dtype=complex)

        # |b>|0>_anc : ancilla=0 means MSB=0
        # Index = 0 * 2^n + int(b, 2) = int("0" + b, 2)
        idx_b0 = int(b, 2)  # ancilla=0, so just the system index
        
        # |b'>|1>_anc : ancilla=1 means MSB=1  
        # Index = 1 * 2^n + int(b', 2) = int("1" + b', 2) = 2^n + int(b', 2)
        idx_bp1 = (1 << num_qubits) + int(b_prime, 2)

        H[idx_b0, idx_b0] = +1   # |b>|0> term
        H[idx_bp1, idx_bp1] = -1 # |b'>|1> term
        
        with np.printoptions(precision=3, suppress=True):
            print("Applying phase correction:")
            print(f"Delta for HW {m}: {delta:.4f} rad")
            print(f"Indices: |b>|0> = {idx_b0}, |b'>|1> = {idx_bp1}")
            print("Hamming weight matrix H:")
            print_nontrivial_unitary_basis_action(H, num_qubits + 1, is_hamiltonian=True)
        
        # Exponentiate
        U_phase = scipy.linalg.expm(1j * delta * H)
        
        with np.printoptions(precision=3, suppress=True, linewidth=1000):
            print("Submatrix to implement:")
            print_nontrivial_unitary_basis_action(U_phase, num_qubits + 1)
            print("Determinant:", np.linalg.det(U_phase))
        
        # Apply the 2-level unitary decomposition
        NQubitDecomposer.apply_2level_nqubit_hw2_unitary(
            qc, U_phase, qargs_with_anc
        )


def su2_AB_from_U(U: np.ndarray, atol: float = 1e-10) -> Tuple[np.ndarray, np.ndarray]:
    """Find A,B ∈ SU(2) such that A B A† B† = U.
    
    This construction follows Eq. (64) from the paper.
    
    Args:
        U: 2x2 unitary matrix
        atol: Absolute tolerance for numerical comparisons
        
    Returns:
        Tuple of (A, B) matrices in SU(2)
        
    Raises:
        ValueError: If U is not 2x2 or has zero determinant
    """
    if U.shape != (2, 2):
        raise ValueError("U must be 2x2")

    # Remove global phase so that det(U_su2) = 1
    detU = np.linalg.det(U)
    # Check that detU is 1 
    if not np.isclose(abs(detU), 1.0, atol=atol):
        raise ValueError(f"Determinant of U must have magnitude 1, got {detU}")
    
    U_su2 = U 

    evals, evecs = np.linalg.eig(U_su2)
    φ1, φ2 = np.angle(evals[0]), np.angle(evals[1])

    θ = 0.5 * (φ1 - φ2)


    # Build exp(i θ Z/2) and exp(i θ Z)
    # Z = np.diag([1.0, -1.0])
    D = np.diag([np.exp(1j * θ), np.exp(-1j * θ)])
    
    W = evecs  # columns are eigenvectors
    
    if not np.allclose(W @ D @ W.conj().T, U_su2, atol=1e-6):
        # Eigenvalues are in opposite order, swap columns of W
        W = W[:, ::-1]
        # Or equivalently, negate θ
        # θ = -θ
        if not np.allclose(W @ D @ W.conj().T, U_su2, atol=1e-6):
            raise ValueError(f"Eigen-decomposition failed to reconstruct U_su2 {U_su2}, got {W @ D @ W.conj().T}, angles {θ}, D = {D}")
        
    exp_iθZ_over2 = np.diag(np.exp(1j * θ * np.array([1.0, -1.0]) / 2.0))
    # A(1) = W exp(i θ Z/2) W†
    A1 = W @ exp_iθZ_over2 @ W.conj().T

    # B(1) = i W X W†, where X = [[0,1],[1,0]]
    X = np.array([[0.0, 1.0], [1.0, 0.0]], dtype=complex)
    B1 = 1j * W @ X @ W.conj().T

    return A1, B1


In [73]:

class NQubitDecomposer:
    """Decomposes n-qubit energy-conserving unitaries."""
    
    @staticmethod
    def decompose_ncontrolled_2level_unitary(U: np.ndarray, control_bits: List[int], control_vals: List[int], target_bits: List[int]) -> List[ControlledTwoLevel]:
        k = len(control_bits)

        if k == 1:
            return [ControlledTwoLevel(U=U, control_bits=control_bits.copy(),
                                      control_vals=control_vals.copy(),
                                      target_bits=target_bits.copy())]
        

        detU = np.linalg.det(U)
        U_su2 = U

        A1, B1 = su2_AB_from_U(U_su2)
        
        assert np.allclose(A1 @ A1.conj().T, np.eye(2)), "A1 is not unitary"
        assert np.allclose(B1 @ B1.conj().T, np.eye(2)), "B1 is not unitary"
        
        assert np.allclose(A1 @ B1 @ A1.conj().T @ B1.conj().T, U_su2), f"SU(2) decomposition failed for {U_su2}"
                

        mid = k // 2
        c1 = control_bits[:mid]
        v1 = control_vals[:mid]
        c2 = control_bits[mid:]
        v2 = control_vals[mid:]

        gates = []
        gates += NQubitDecomposer.decompose_ncontrolled_2level_unitary(A1, c1, v1, target_bits)
        gates += NQubitDecomposer.decompose_ncontrolled_2level_unitary(B1, c2, v2, target_bits)
        gates += NQubitDecomposer.decompose_ncontrolled_2level_unitary(A1.conj().T, c1, v1, target_bits)
        gates += NQubitDecomposer.decompose_ncontrolled_2level_unitary(B1.conj().T, c2, v2, target_bits)
        
        return gates
    
    @staticmethod
    def apply_2level_nqubit_hw2_unitary(qc: QuantumCircuit, U: np.ndarray, 
                                   qargs: List[int], hw_indices: List[int] = None, tlb=None):
        n = len(qargs)
        dim = 2**n
        
        # skip if U is identity
        if np.allclose(U, np.eye(dim), atol=1e-12):
            print("Input unitary is identity; skipping decomposition.")
            return
        
        local_i, local_j = get_nontrivial_rows_cols(U)
        # local_j, local_i = get_nontrivial_rows_cols(U)
        
        # if tlb is not None:
        #     local_i, local_j = tlb.i, tlb.j
        
        # print(f"local_i={local_i}, local_j={local_j}")
                
        submatrix = np.array([
            [U[local_i, local_i], U[local_i, local_j]],
            [U[local_j, local_i], U[local_j, local_j]]
        ], dtype=complex)
        
        is_diag = np.isclose(submatrix[0,1], 0.0, atol=1e-12) and np.isclose(submatrix[1,0], 0.0, atol=1e-12)
        
        # submatrix = -1 * submatrix  # Adjust global phase for consistency
        
            # Make it SU(2): factor out determinant as a scalar phase
        detU = np.linalg.det(submatrix)
        if not np.isclose(np.real(detU), 1.0, atol=1e-10):
            raise ValueError(f"2-level submatrix determinant not 1: det={detU}")
        su2_block = submatrix   # scalar goes into D, not Q
        
        # with np.printoptions(precision=3, suppress=True, linewidth=120):
        #     print(f"  Extracted 2-level SU(2) block (det {detU:.3f}):\n {su2_block}")
        

        # Determine control bits and target bits
        bits_i = bin(local_i)[2:].zfill(n)
        bits_j = bin(local_j)[2:].zfill(n)
        # print(f"  Applying TLU: |{bits_i}⟩ ↔ |{bits_j}⟩ on local indices {local_i}, {local_j}")
        hd = sum(b1 != b2 for b1, b2 in zip(bits_i, bits_j))
        # print(f"  [apply_2level_nqubit_hw2_unitary] local_i={local_i} |{bits_i}⟩, local_j={local_j} |{bits_j}⟩, HD={hd}")
    

        control_bits = []
        control_vals = []
        target_bits = []
        
        for bit in range(n):
            bit_i = (local_i >> bit) & 1
            bit_j = (local_j >> bit) & 1
            if bit_i == bit_j:
                control_bits.append(bit)
                control_vals.append(bit_i)
            else:
                target_bits.append(bit)
        
        # print(f"    Target bits differ at positions: {target_bits} and control bits: {control_bits} with values {control_vals}")

        if len(target_bits) != 2:
            raise ValueError(f"2-level unitary does not differ in exactly 2 bit positions. target_bits={target_bits}, control_bits={control_bits}")

        print(f"    Controls: {list(zip(control_bits, control_vals))}, Targets: {target_bits}")

        # Decompose into single-controlled gates
        controlled_blocks = NQubitDecomposer.decompose_ncontrolled_2level_unitary(
            su2_block, control_bits, control_vals, target_bits
        )
        
        #         # Helper: map pattern on two targets to basis index (order: [t0, t1])
        def basis_index_on_targets(pat):
            return (int(pat[0]) << 1) + int(pat[1])

        # Implement each controlled block
        for block in controlled_blocks:
            
            with np.printoptions(precision=3, suppress=True):
                print("  Controlled block info, its determinant")
                print(f"  Controlled block U:\n {block.U}")
                print(f"  Determinant: {np.linalg.det(block.U):.3f}")
                
            
            theta, phi, lam, global_phase = OneQubitEulerDecomposer('XYX').angles_and_phase(
                Operator(block.U)
            )

            detU = np.linalg.det(block.U)

            # Normalize phase into (-π, π]
            phase = (global_phase + np.pi) % (2 * np.pi) - np.pi

            if np.isclose(detU, 1.0, atol=1e-8):
                if np.isclose(phase, 0.0, atol=1e-8):
                    # Already SU(2) up to numerical noise
                    print(f"    Det≈1, phase≈0; θ={theta:.3f}, φ={phi:.3f}, λ={lam:.3f}")
                    pass
                elif np.isclose(abs(phase), np.pi, atol=1e-8):
                    # U is - (Rx Ry Rx); absorb -1 into one rotation using 2π periodicity
                    print("    Det≈1 but global phase ≈ π; absorbing into θ.")
                    theta += 2 * np.pi          # could also adjust φ or λ instead
                    global_phase = 0.0
                    
                    # fix angles into standard range
                    # theta = (theta + np.pi) % (2 * np.pi) - np.pi
                    # print(f"    Adjusted angles: θ={theta:.3f}, φ={phi:.3f}, λ={lam:.3f}, phase={global_phase:.3f}")
                else:
                    # In exact math this shouldn't happen if det(U)=1; probably numerical issues
                    print(f"    Warning: det≈1 but unexpected global phase {phase:.6f}")
            else:
                print(f"    det(U) ≈ {detU}, leaving global phase as given: {global_phase:.6f}")


            if len(block.control_bits) != 1:
                raise RuntimeError(
                    "Final decomposition should only contain single-controlled gates."
                )

            ctrl_bit = block.control_bits[0]
            ctrl_val = block.control_vals[0]
            t0_bit, t1_bit = block.target_bits

            control = qargs[ctrl_bit]
            target1 = qargs[t0_bit]
            target2 = qargs[t1_bit]
            
            # if is_diag:
            #     theta = -theta

            # GateSynthesizer.controlled_exp_i_theta_L(qc, theta/2, ctrl_val, target1, target2, control)
            GateSynthesizer.controlled_exp_i_alpha_R(qc, -phi/2, ctrl_val, target1, target2, control)
            GateSynthesizer.controlled_exp_i_theta_L(qc, -theta/2, ctrl_val, target1, target2, control)
            GateSynthesizer.controlled_exp_i_alpha_R(qc, -lam/2, ctrl_val, target1, target2, control)
      
         
    @staticmethod
    def decompose_general_unitary(qc: QuantumCircuit, U: np.ndarray, qargs: List[int]) -> Dict[int, List[float]]:
        n_qubits = len(qargs)
        dim = 2**n_qubits

        assert U.shape == (dim, dim), f"Unitary shape {U.shape} incompatible with {n_qubits} qubits."
        # assert np.isclose(np.linalg.det(U), 1.0, atol=1e-10), "Unitary must have determinant 1."
        assert np.allclose(U.conj().T @ U, np.eye(dim), atol=1e-10), "Unitary must be unitary."
        assert is_energy_conserving(U), "Unitary must be energy-conserving."


        hw_to_indices: Dict[int, List[int]] = {}
        for i in range(dim):
            hw = bin(i).count("1")
            hw_to_indices.setdefault(hw, []).append(i)

        hw_phases: Dict[int, List[float]] = {} # Store phases for each Hamming weight sector
        
        blocks = get_hamming_weight_blocks(U)

        for hw in sorted(hw_to_indices.keys()):
            indices = hw_to_indices[hw]
            block_size = len(indices)
            block = blocks[hw]
            
            assert block_size == block.shape[0], "Block size mismatch"
            assert block_size == block.shape[1], "Block size mismatch"

            if block_size == 1:
                phase = np.angle(block[0, 0])
                hw_phases[hw] = phase
                continue
                        
            det_block = np.linalg.det(block)
            D = np.eye(block_size, dtype=complex)
            D[0,0] = np.exp(-1j * np.angle(det_block))  # Factor out global phase
            su2_block = D @ block  # Remove global phase
            # D = np.diag([np.exp(-1j * np.angle(det_block) / block_size)] * block_size)
            # su2_block = D @ block  # Normalize to SU(2)
            # su2_block = block / np.pow(det_block, 1/block.shape[0])
            phase = np.angle(det_block)
                    
            hw_phases[hw] = phase # Total phase for this HW sector
            block = su2_block
            
            
            assert np.allclose(su2_block.conj().T @ su2_block, np.eye(block_size), atol=1e-10), "Normalized block is not unitary"
            assert np.isclose(np.linalg.det(su2_block), 1.0, atol=1e-10), "Normalized block does not have determinant 1"
            
            if np.allclose(block, np.eye(block_size), atol=1e-10):
                print(f"  HW={hw} block is identity; skipping.")
                continue
            
            print(f"\nDecomposing Hamming weight {hw} block of size {block_size} with phase {phase:.6f} rad:")
            print_nontrivial_unitary_basis_action(block, block_size)
            
            two_level_blocks = extract_two_level_blocks(block, atol=1e-10, enforce_hamming_distance=None)
            if not (two_level_blocks):    
                Us = sud_decompose(block)
                two_level_blocks = []
                prod = np.eye(block_size, dtype=complex)
                with np.printoptions(precision=3, suppress=True):
                    for tlu in Us:
                        print(tlu.to_full_matrix())
                        two_level_blocks.append(
                            tlu
                        )
                        prod = prod @ tlu.to_full_matrix() 
                # reverse the order to match application order
                # two_level_blocks = list(reversed(two_level_blocks))
                
                print("Reconstructed block from SUD decomposition:")
                print_nontrivial_unitary_basis_action(prod, block_size)
                assert np.allclose(prod, block, atol=1e-8), "Reconstructed block does not match original"
                # verify the decomposition
                
            for tlb in two_level_blocks:                
                
                bits_i = bin(indices[tlb.i])[2:].zfill(n_qubits)
                bits_j = bin(indices[tlb.j])[2:].zfill(n_qubits)
                
                hd = sum(b1 != b2 for b1, b2 in zip(bits_i, bits_j))
                print(f"    Hamming distance between |{bits_i}⟩ and |{bits_j}⟩: {hd}")
                
                global_i = indices[tlb.i]
                global_j = indices[tlb.j]
                print(f"    Applying 2-level unitary between global indices {global_i} and {global_j} and tlb.i={tlb.i}, tlb.j={tlb.j}")
                
                if hd == 2:
                    full_U = np.eye(dim, dtype=complex)
                    full_U[global_i, global_i] = tlb.submatrix[0, 0]
                    full_U[global_i, global_j] = tlb.submatrix[0, 1]
                    full_U[global_j, global_i] = tlb.submatrix[1, 0]
                    full_U[global_j, global_j] = tlb.submatrix[1, 1]
                    
                    NQubitDecomposer.apply_2level_nqubit_hw2_unitary(qc, full_U, qargs, indices)
                     # implemented till now
                    U_now = Operator(qc).data
                    # extract current block
                    current_blocks = get_hamming_weight_blocks(U_now)
                    current_block = current_blocks[hw]
                    print("\nCurrent block in circuit:")
                    print_nontrivial_unitary_basis_action(current_block, block_size)
                    
                elif hd > 2:
                    
                    conj_gates, b_double_prime = construct_conjugation_gates(
                        global_i, global_j, n_qubits
                    )
                    for ctrl, t1, t2 in conj_gates:
                        apply_controlled_iswap(qc, qargs[ctrl], qargs[t1], qargs[t2])
                    
                    # Construct W = K V K† acting on (b'', b')
                    # According to Eq. (76) of the paper
                    t = hd // 2  # Total number of swaps needed
                    
                    # Phase factors from Eq. (76): ̃U = diag(i^(t-1), 1) · U · diag(i^(-(t-1)), 1)
                    phase_factor_upper = (1j) ** (t - 1)      # i^(t-1)
                    phase_factor_lower = (1j) ** (1 - t)      # i^(1-t)
                    
                    
                    W_matrix = np.array([
                        [tlb.submatrix[0, 0], phase_factor_upper * tlb.submatrix[0, 1]],
                        [phase_factor_lower * tlb.submatrix[1, 0], tlb.submatrix[1, 1]]
                    ], dtype=complex)
                    
                    # Apply W (which has Hamming distance 2)
                    full_W = np.eye(dim, dtype=complex)
                    full_W[b_double_prime, b_double_prime] = W_matrix[0, 0]
                    full_W[b_double_prime, global_j] = W_matrix[0, 1]
                    full_W[global_j, b_double_prime] = W_matrix[1, 0]
                    full_W[global_j, global_j] = W_matrix[1, 1]

                    
                    NQubitDecomposer.apply_2level_nqubit_hw2_unitary(qc, full_W, qargs, indices)
                    
                    for ctrl, t1, t2 in reversed(conj_gates):
                        apply_controlled_iswap(qc, qargs[ctrl], qargs[t1], qargs[t2], inverse=True)
                        
                else:
                    print(f"    Skipping: Hamming distance = {hd}")
                    continue
        
        print("\nFixing relative phases between Hamming weight sectors...")
        print(f"  HW phases: {hw_phases}")

        fix_relative_phases(
            qc, qargs,
            [hw_phases.get(i, 0.0) for i in range(n_qubits + 1)]
            
        )       
        return hw_phases
        


In [68]:
def make_3q_two_level_unitary(n_qubits=4, i=1, j=2, theta=np.pi/2, phi=np.pi/4):
    theta = theta
    phi = phi

    c = np.cos(theta)
    s = np.sin(theta)

    # sub = np.array([[c,  np.exp(1j * phi) * s],
    #                 [-np.exp(-1j * phi) * s, c]],dtype=complex)
    # sub = np.array(
    #     [
    #         [0, 1j],
    #         [-1j, 0]
    #     ],
    #     dtype=complex
    # )
    sub = np.array(
        [
            [c+1j*s, 0],
            [0, c-1j*s]
        ],
        dtype=complex
    )
    U = np.eye(2**n_qubits, dtype=complex)

    U[i, i] = sub[0, 0]
    U[i, j] = sub[0, 1]
    U[j, i] = sub[1, 0]
    U[j, j] = sub[1, 1]
    
    theta_x, phi_y, lam_x, global_phase = OneQubitEulerDecomposer('XYX').angles_and_phase(
        Operator(sub)
    )
    with np.printoptions(precision=3, suppress=True):
        print("Submatrix:")
        print(sub)
        print(f"Euler angles (XYX) of submatrix: theta={theta_x}, phi={phi_y}, lambda={lam_x}, global_phase={global_phase}")
    return U


def two_state_opposite_phase_unitary(n, i, j, alpha):
    dim = 2**n
    U = np.eye(dim, dtype=complex)
    U[i, i] = np.exp(1j * alpha)   # phase on |i>
    U[j, j] = np.exp(-1j * alpha)  # opposite phase on |j>
    return U


def test_simple_2level_hd2(theta, phi, i, j):
    n_qubits = 3
    dim = 2**n_qubits
        
    # U_target = make_3q_two_level_unitary(n_qubits=n_qubits, i=i, j=j, theta=theta, phi=phi)
    U_target = two_state_opposite_phase_unitary(n_qubits, i, j, theta)
    
    print(f"Target unitary (dim={U_target.shape}) (2-level, HD=2):")
    print_nontrivial_unitary_basis_action(U_target, n_qubits)
      
    # Decompose it
    qc = QuantumCircuit(n_qubits)
    NQubitDecomposer.decompose_general_unitary(qc, U_target, list(range(n_qubits)))
    
    # Check result
    U_impl = Operator(qc).data
    
    print("\nImplemented unitary:")
    print_nontrivial_unitary_basis_action(U_impl, n_qubits+1)

    U_impl = get_effective_unitary(qc, ancilla_indices=[n_qubits], ancilla_state=0)
    print(f"Shapes: U_target {U_target.shape}, U_impl {U_impl.shape}")
    match = _unitaries_close_up_to_phase(U_target, U_impl, atol=1e-6)
    print(f"\nMatch: {match}")
    
    if not match:
            print("\nImplemented unitary debug info:")
            print_nontrivial_unitary_basis_action(U_impl, n_qubits)
    
    assert match, "HD=2 case should work!"
    print("✓ Test passed")
    return match

test_simple_2level_hd2(theta=np.pi/3, phi=0, i=2, j=4)

Target unitary (dim=(8, 8)) (2-level, HD=2):
|010> -> (+0.500+0.866j)|010>
|100> -> (+0.500-0.866j)|100>

Decomposing Hamming weight 1 block of size 3 with phase 0.000000 rad:
|1> -> (+0.500+0.866j)|1>
|10> -> (+0.500-0.866j)|10>
[[ 1.-0.j -0.-0.j  0.+0.j]
 [ 0.+0.j  1.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  1.+0.j]]
[[ 1.+0.j  0.+0.j -0.-0.j]
 [ 0.+0.j  1.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  1.-0.j]]
[[1. +0.j    0. +0.j    0. +0.j   ]
 [0. +0.j    0.5+0.866j 0. +0.j   ]
 [0. +0.j    0. +0.j    0.5-0.866j]]
Reconstructed block from SUD decomposition:
|1> -> (+0.500+0.866j)|1>
|10> -> (+0.500-0.866j)|10>
    Hamming distance between |001⟩ and |010⟩: 2
    Applying 2-level unitary between global indices 1 and 2 and tlb.i=0, tlb.j=1
Input unitary is identity; skipping decomposition.

Current block in circuit:
    Hamming distance between |001⟩ and |100⟩: 2
    Applying 2-level unitary between global indices 1 and 4 and tlb.i=0, tlb.j=2
Input unitary is identity; skipping decomposition.

Current blo

True

In [74]:
def make_3q_two_level_unitary(n_qubits=4, i=1, j=2):
    """
    Construct an 8x8 unitary that is identity except for a 2x2 SU(2) block
    mixing |001> (index 1) and |010> (index 2).

    Returns:
        U: 8x8 np.ndarray complex
    """
    # Generic SU(2) block
    theta = 3 * np.pi / 2
    phi = 3 * np.pi / 2

    c = np.cos(theta)
    s = np.sin(theta)

    
    sub = np.array(
        [
            [0, 1j],
            [-1j, 0]
        ],
        dtype=complex
    )

    # Full 3-qubit unitary: identity except this 2x2 block
    U = np.eye(2**n_qubits, dtype=complex)

    # basis: |000>=0, |001>=1, |010>=2, |011>=3,
    #        |100>=4, |101>=5, |110>=6, |111>=7
    # i = 1  # |001>
    # j = 2  # |010>

    U[i, i] = sub[0, 0]
    U[i, j] = sub[0, 1]
    U[j, i] = sub[1, 0]
    U[j, j] = sub[1, 1]
    
    # print the euler angles of the submatrix
    theta_x, phi_y, lam_x, global_phase = OneQubitEulerDecomposer('XYX').angles_and_phase(
        Operator(sub)
    )
    with np.printoptions(precision=3, suppress=True):
        print("Submatrix:")
        print(sub)

    return U

def test_simple_2level_hd2():
    n_qubits = 3
    dim = 2**n_qubits
        
    # U_target = make_3q_two_level_unitary(n_qubits=n_qubits, i=1, j=2)
    U_target = make_hw1_cyclic_permutation_unitary(n_qubits=n_qubits)

    print(f"Target unitary (dim={U_target.shape}) (2-level, HD=2) (determinant {np.linalg.det(U_target):.3f}):")
    print_nontrivial_unitary_basis_action(U_target, n_qubits)
      
    # Decompose it
    qc = QuantumCircuit(n_qubits)
    NQubitDecomposer.decompose_general_unitary(qc, U_target, list(range(n_qubits)))
    
    # Check result
    U_impl = Operator(qc).data
    
    print("\nImplemented unitary:")
    print_nontrivial_unitary_basis_action(U_impl, n_qubits+1)
    # with np.printoptions(precision=3, suppress=True, linewidth=200):
    #     print("Real part:")
    #     print(U_impl.real)
    #     print("\nImaginary part:")
    #     print(U_impl.imag)
    
    U_impl = get_effective_unitary(qc, ancilla_indices=[n_qubits], ancilla_state=0)
    print(f"Shapes: U_target {U_target.shape}, U_impl {U_impl.shape}")
    match = _unitaries_close_up_to_phase(U_target, U_impl, atol=1e-6)
    print(f"\nMatch: {match}")
    
    if not match:
        with np.printoptions(precision=3, suppress=True, linewidth=200):
            print("\nImplemented unitary debug info:")
            print_nontrivial_unitary_basis_action(U_impl, n_qubits)
    
    # Get hw blocks and print their phases - theta0
    hw_blocks = get_hamming_weight_blocks(U_impl)
    for hw, block in hw_blocks.items():
        phase = np.angle(np.linalg.det(block)) 
        print(f"  HW={hw} block phase: {phase:.6f} rad")    

    assert match, "HD=2 case should work!"
    print("✓ Test passed")

test_simple_2level_hd2()

Target unitary (dim=(8, 8)) (2-level, HD=2) (determinant -1.000+0.000j):
|001> -> +1.000|010>
|010> -> +1.000|001>

Decomposing Hamming weight 1 block of size 3 with phase 3.141593 rad:
|0> -> +1.000|1>
|1> -> -1.000|0>
Found 2-level block on indices 0,1
2-level block on indices 0,1 has determinant 1.000+0.000j
    Hamming distance between |001⟩ and |010⟩: 2
    Applying 2-level unitary between global indices 1 and 2 and tlb.i=0, tlb.j=1
    Controls: [(2, 0)], Targets: [0, 1]
  Controlled block info, its determinant
  Controlled block U:
 [[ 0.+0.j -1.-0.j]
 [ 1.+0.j  0.+0.j]]
  Determinant: 1.000+0.000j
    Det≈1, phase≈0; θ=3.142, φ=1.571, λ=1.571

Current block in circuit:
|0> -> +1.000|1>
|1> -> -1.000|0>
  HW=2 block is identity; skipping.

Fixing relative phases between Hamming weight sectors...
  HW phases: {0: np.float64(0.0), 1: np.float64(3.141592653589793), 2: np.float64(0.0), 3: np.float64(0.0)}

Fixing relative phases: [np.float64(0.0), np.float64(3.141592653589793), np.f

In [76]:
from qiskit.circuit.library import SwapGate

def make_exp_i_theta_swap12_swap34(theta):
    """
    Construct U = exp(i θ (SWAP_01 · SWAP_23)) for 4 qubits.
    Uses S^2 = I => exp(i θ S) = cos θ I + i sin θ S.
    """
    # 4-qubit SWAP_01 followed by SWAP_23
    qc_swap = QuantumCircuit(4)
    qc_swap.append(SwapGate(), [0, 1])
    qc_swap.append(SwapGate(), [2, 3])
    S = Operator(qc_swap).data  # 16x16 involution

    dim = S.shape[0]
    I = np.eye(dim, dtype=complex)

    U = np.cos(theta) * I + 1j * np.sin(theta) * S
    return U


def make_exp_i_theta_swap12(theta):
    """
    Construct U = exp(i θ SWAP_01) for 4 qubits.
    Uses S^2 = I => exp(i θ S) = cos θ I + i sin θ S.
    """
    # 4-qubit SWAP_01
    qc_swap = QuantumCircuit(4)
    qc_swap.append(SwapGate(), [0, 1])
    S = Operator(qc_swap).data  # 16x16 involution

    dim = S.shape[0]
    I = np.eye(dim, dtype=complex)

    U = np.cos(theta) * I + 1j * np.sin(theta) * S
    return U

def test_decompose_exp_i_theta_swap12_swap34():
    # choose some nontrivial angle
    theta = np.pi / 2

    # 1) Ideal target unitary
    U_target = make_exp_i_theta_swap12(theta)

    with np.printoptions(precision=3, suppress=True, linewidth=120):
        print("Target U:\n", U_target.imag)

    qc_impl = QuantumCircuit(2)
    # qargs in order [0,1,2,3]
    hw_phases = NQubitDecomposer.decompose_general_unitary(qc_impl, U_target, [0, 1])

    U_impl = Operator(qc_impl).data
    U_impl = get_effective_unitary(qc_impl, ancilla_indices=[2], ancilla_state=0)

    # 3) Print both matrices (rounded) for inspection
    with np.printoptions(precision=3, suppress=True, linewidth=120):
        print("=== Expected U (exp(i θ SWAP_01 SWAP_23)) ===")
        # print(U_target.imag)
        print_nontrivial_unitary_basis_action(U_target, n_qubits=4)
        print("\n=== Implemented U (from EC decomposer) ===")
        # print(U_impl.imag)
        print_nontrivial_unitary_basis_action(U_impl, n_qubits=4)
        print("\n=== Implemented U (real part) ===")
        print(U_impl.real)
        # print("\n=== Implemented U (real part) ===")
        # print(U_impl.real)

    # 4) Check they match up to global phase
    assert _unitaries_close_up_to_phase(U_impl, U_target), \
        "exp(i θ SWAP_01 SWAP_23) decomposition failed!"


# ---------------------------------------------------------
# Run directly
# ---------------------------------------------------------
test_decompose_exp_i_theta_swap12_swap34()

Target U:
 [[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]


AssertionError: Unitary shape (16, 16) incompatible with 2 qubits.

In [ ]:
def test_fix_relative_phases():
    """Test that fix_relative_phases applies correct diagonal phases per HW sector."""
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Operator
    import numpy as np
    
    num_qubits = 3
    qc = QuantumCircuit(num_qubits)
    qargs = list(range(num_qubits))
    
    # Phases: HW=0 -> 0, HW=1 -> π/4, HW=2 -> π/2, HW=3 -> π
    phases = [0.0, np.pi/4, np.pi/2, np.pi]
    
    fix_relative_phases(qc, qargs, phases)
    
    # Extract unitary (ignoring ancilla - project onto ancilla=0)
    U_full = Operator(qc).data
    n_total = qc.num_qubits
    n_ancilla = qc.num_ancillas
    
    # Get effective unitary on logical qubits (ancilla in |0⟩)
    dim_logical = 2**num_qubits
    logical_indices = [i for i in range(2**n_total) if (i >> num_qubits) == 0]  # ancilla bits = 0
    U_eff = U_full[np.ix_(logical_indices, logical_indices)]
    
    # Check: each basis state |i⟩ should get phase exp(i*(theta_hw - theta_0))
    theta_0 = phases[0]
    for i in range(dim_logical):
        hw = bin(i).count('1')
        expected_phase = np.exp(-1j * (phases[hw] - theta_0))
        actual_phase = U_eff[i, i]
        
        # assert np.isclose(np.abs(actual_phase), 1.0, atol=1e-6), \
        #     f"State {i}: not a phase, got |{actual_phase}|"
        # assert np.isclose(actual_phase, expected_phase, atol=1e-6), \
        #     f"State {i} (HW={hw}): expected {expected_phase:.3f}, got {actual_phase:.3f}"
        
        print(f"State |{i}⟩ (HW={hw}): expected phase {expected_phase:.3f}, actual phase {actual_phase:.3f}")
        
        # Off-diagonal should be zero (diagonal unitary)
        for j in range(dim_logical):
            if i != j:
                assert np.abs(U_eff[i, j]) < 1e-6, \
                    f"Off-diagonal U[{i},{j}] = {U_eff[i,j]:.3f} should be 0"
    
    print("✓ All phase corrections verified!")

test_fix_relative_phases()


Fixing relative phases: [0.0, 0.7853981633974483, 1.5707963267948966, 3.141592653589793]

Fixing relative phase for Hamming weight 1: delta = 0.7854 rad
 |b> = |001>, |b'> = |000>
Applying phase correction:
Delta for HW 1: 0.7854 rad
Indices: |b>|0> = 1, |b'>|1> = 8
Hamming weight matrix H:
|0001> -> +1.000|0001>
|1000> -> -1.000|1000>
Submatrix to implement:
|0001> -> (+0.707+0.707j)|0001>
|1000> -> (+0.707-0.707j)|1000>
Determinant: (1+1.0146536357569526e-17j)
  Extracted 2x2 submatrix for local indices 1, 8:
[[0.707+0.707j 0.   +0.j   ]
 [0.   +0.j    0.707-0.707j]]
  Applying TLU: |0001⟩ ↔ |1000⟩ on local indices 1, 8
  [apply_2level_nqubit_hw2_unitary] local_i=1 |0001⟩, local_j=8 |1000⟩, HD=2
    Target bits differ at positions: [0, 3] and control bits: [1, 2] with values [0, 0]
    Controls: [(1, 0), (2, 0)], Targets: [0, 3]
  Controlled block U:
 [[0.924+0.383j 0.   +0.j   ]
 [0.   +0.j    0.924-0.383j]]
  Determinant: 1.000-0.000j
    Det≈1, phase≈0; θ=0.785, φ=-1.571, λ=1.571

In [5]:
def make_hw1_cyclic_permutation_unitary(n_qubits=3):
    """
    Construct a 2^n x 2^n unitary that acts as a cyclic permutation (123)
    on the Hamming weight 1 sector and identity elsewhere.
    
    For n=3 qubits, the HW=1 basis states are:
      |001> (index 1), |010> (index 2), |100> (index 4)
    
    The cyclic permutation (123) acts as:
      |001> -> |010>
      |010> -> |100>
      |100> -> |001>
    
    This is a 3-cycle with determinant 1 (even permutation).
    
    Returns:
        U: 2^n x 2^n np.ndarray complex unitary
    """
    dim = 2**n_qubits
    U = np.eye(dim, dtype=complex)
    
    # HW=1 indices for n=3: |001>=1, |010>=2, |100>=4
    # Cyclic permutation: 1 -> 2 -> 4 -> 1
    hw1_indices = [1, 2, 4]  # |001>, |010>, |100>
    
    # Clear the diagonal entries for HW=1 states
    for idx in hw1_indices:
        U[idx, idx] = 0
    
    # Apply cyclic permutation: (1 -> 2 -> 4 -> 1)
    # |001> -> |010>: U[2, 1] = 1
    # |010> -> |100>: U[4, 2] = 1
    # |100> -> |001>: U[1, 4] = 1
    # U[2, 1] = 1  # |001> maps to |010>
    # U[4, 2] = 1  # |010> maps to |100>
    # U[1, 4] = 1  # |100> maps to |001>
    
    U[1, 2] = 1  # |010> maps to |001>
    U[2, 1] = 1  # |001> maps to |010>
    U[4, 4] = 1  # |100> maps to |100> (identity, keep the diagonal)
    
    return U
